# Anti-Helmholtz Coils Simulation
This document provides comprehensive analysis and design tools for anti-Helmholtz coil configurations, commonly used in magneto-optical traps (MOTs) and atomic physics experiments.
Includes interactive matplotlib widgets that let you vary key parameters such as coil radius, separation, current, and number of turns to study the resulting magnetic field profiles.

## Brief Theory Introduction

Anti-Helmholtz coils are a pair of circular coils carrying current in opposite directions, separated by a distance typically equal to their radius. Unlike Helmholtz coils which produce a uniform magnetic field at the center, anti-Helmholtz coils create a magnetic field minimum at the center point, with the field increasing linearly in all directions away from the center.

### Magnetic Field from a Single Circular Loop

The magnetic field at a point along the axis of a circular loop of radius $R$ carrying current $I$ at a distance $z$ from the center is:
$$B_z(z) = \frac{\mu_0 I R^2}{2 \left(R^2 + z^2\right)^{3/2}}$$

where $\mu_0 = 4\pi \times 10^{-7}$ T m/A is the magnetic permeability of vacuum.

The radial component is:
$$B_r(z) = \frac{\mu_0 I R z}{2 \left(R^2 + z^2\right)^{3/2}}$$

### Anti-Helmholtz Configuration

For anti-Helmholtz coils, two coils separated by distance $d$ have currents flowing in opposite directions. The field at position $z$ is:
$$B_z^{\mathrm{total}}(z) = B_z^{(1)}\left(z + \frac{d}{2}\right) - B_z^{(2)}\left(z - \frac{d}{2}\right)$$

At the center ($z = 0$), $B_z(0) = 0$, and the field varies linearly near the center:
$$B_z(z) \approx \frac{3 \mu_0 I R^2 a}{2 \left(R^2 + a^2\right)^{5/2}} \cdot 2z = b' \cdot 2z$$

where $a=d/2$ and $b'$ is the magnetic field gradient (in T/m).

### Quadrupole Approximation

The 3D magnetic field near the center forms a quadrupole field:
$$\mathbf{B}(x, y, z) \approx b' \begin{pmatrix} x \\ y \\ -2z \end{pmatrix}.$$

### Key Properties

* Field minimum at center: $B(0,0,0) = 0$
* Linear gradient: field increases linearly away from center
* Quadrupole symmetry: field lines point toward center from all directions
* Used in MOTs: provides restoring force for trapped atoms

### Constants and Setup
Here we define physical constants and import necessary libraries.

In [1]:
%matplotlib inline

import math
import numpy as np
from scipy import constants
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as w
from IPython.display import display, clear_output

mu_0 = 4 * math.pi * 1e-7
hbar = constants.hbar
amu = 1.66053906660e-27
mu_B = 9.274009994e-24

cmap_field = plt.cm.RdBu_r

## Single Circular Loop Magnetic Field

Building block of anti-Helmholtz configuration.

In [2]:
def B_single_loop(R, I, z, N=1):
    z = np.asarray(z)
    R2 = R**2
    denominator = (R2 + z**2)**(3/2)
    Bz = (mu_0 * I * N * R2) / (2 * denominator)
    Br = (mu_0 * I * N * R * z) / (2 * denominator)
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def plot_single_loop(R=0.1, I=10.0, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz, Br, B_total = B_single_loop(R, I, z, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(z * 100, Bz * 1e4, label='Bz', color='blue', lw=2)
    ax1.plot(z * 100, Br * 1e4, label='Br', color='orange', lw=2, ls='--')
    ax1.plot(z * 100, B_total * 1e4, label='|B|', color='red', lw=2)
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Single Loop: R={R*100:.1f} cm, I={I:.1f} A, N={N}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', lw=0.5)
    
    zoom_range = 0.05
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz[mask] * 1e4, label='Bz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, B_total[mask] * 1e4, label='|B|', color='red', lw=2)
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Zoom: |z| <= {zoom_range*100:.1f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()

R_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
N_slider = w.IntSlider(description='Turns', value=1, min=1, max=50, step=1)
z_max_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

@w.interact(R=R_slider, I=I_slider, N=N_slider, z_max=z_max_slider)
def update_single_loop(R, I, N, z_max):
    plot_single_loop(R/100, I, N, z_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## Anti-Helmholtz Coils Configuration

Two coils with opposite currents creating field minimum at center.

In [3]:
def B_anti_helmholtz(R, I, d, z, N=1):
    z = np.asarray(z)
    z1 = z + d/2
    Bz1, Br1, _ = B_single_loop(R, I, z1, N)
    z2 = z - d/2
    Bz2, Br2, _ = B_single_loop(R, -I, z2, N)
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def compute_gradient_at_center(R, I, d, N=1):
    dz = 1e-6
    Bz_pos, _, _ = B_anti_helmholtz(R, I, d, np.array([dz]), N)
    Bz_neg, _, _ = B_anti_helmholtz(R, I, d, np.array([-dz]), N)
    gradient = (Bz_pos[0] - Bz_neg[0]) / (2 * dz)
    return gradient

def plot_anti_helmholtz(R=0.1, I=10.0, d=0.1, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz, Br, B_total = B_anti_helmholtz(R, I, d, z, N)
    gradient = compute_gradient_at_center(R, I, d, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(z * 100, Bz * 1e4, label='Bz', color='blue', lw=2)
    ax1.plot(z * 100, B_total * 1e4, label='|B|', color='red', lw=2)
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Anti-Helmholtz: R={R*100:.1f} cm, I={I:.1f} A, N={N}, d/R={d/R:.2f}')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', lw=0.5)
    
    zoom_range = 0.03
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz[mask] * 1e4, label='Actual Bz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, gradient * z[mask] * 1e4, 
             label=f'Linear approx: bprime = {abs(gradient)*1e2:.2f} G/cm', 
             color='green', lw=2, ls='--')
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Zoom: |z| <= {zoom_range*100:.1f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', lw=0.5)
    plt.tight_layout()
    plt.show()
    return gradient

R_ah_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_ah_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_ah_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_ah_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
z_max_ah_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

gradient_output = w.Output()

@w.interact(R=R_ah_slider, I=I_ah_slider, d=d_ah_slider, N=N_ah_slider, z_max=z_max_ah_slider)
def update_anti_helmholtz(R, I, d, N, z_max):
    gradient = plot_anti_helmholtz(R/100, I, d/100, N, z_max/100)
    with gradient_output:
        clear_output()
        print(f'Gradient: {abs(gradient)*1e2:.4f} G/cm')
        if abs(d - R) < 0.01:
            print('Ideal anti-Helmholtz configuration (d = R)')

display(gradient_output)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

Output()

## Comparison: Helmholtz vs Anti-Helmholtz

Side-by-side comparison of both configurations.

In [ ]:
def B_helmholtz(R, I, d, z, N=1):
    z = np.asarray(z)
    z1 = z + d/2
    Bz1, Br1, _ = B_single_loop(R, I, z1, N)
    z2 = z - d/2
    Bz2, Br2, _ = B_single_loop(R, I, z2, N)
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Bz, Br, B_total

def plot_comparison(R=0.1, I=10.0, d=0.1, N=1, z_max=0.3):
    z = np.linspace(-z_max, z_max, 500)
    Bz_ah, _, _ = B_anti_helmholtz(R, I, d, z, N)
    gradient_ah = compute_gradient_at_center(R, I, d, N)
    Bz_h, _, _ = B_helmholtz(R, I, d, z, N)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(z * 100, Bz_h * 1e4, label='Helmholtz Bz', color='blue', lw=2)
    ax1.plot(z * 100, Bz_ah * 1e4, label='Anti-Helmholtz Bz', color='red', lw=2, ls='--')
    ax1.set_xlabel('z (cm)')
    ax1.set_ylabel('B (G)')
    ax1.set_title(f'Comparison: R={R*100:.0f} cm, d={d*100:.0f} cm')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    zoom_range = 0.05
    mask = np.abs(z) <= zoom_range
    ax2.plot(z[mask] * 100, Bz_h[mask] * 1e4, label='Helmholtz', color='blue', lw=2)
    ax2.plot(z[mask] * 100, Bz_ah[mask] * 1e4, label='Anti-Helmholtz', color='red', lw=2, ls='--')
    ax2.set_xlabel('z (cm)')
    ax2.set_ylabel('B (G)')
    ax2.set_title(f'Center: |z| <= {zoom_range*100:.0f} cm')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    B_center_h = Bz_h[np.argmin(np.abs(z))]
    print(f'Helmholtz: B_center = {B_center_h*1e4:.4f} G (uniform)')
    print(f'Anti-Helmholtz: B_center = {Bz_ah[np.argmin(np.abs(z))]*1e4:.4f} G, Gradient = {abs(gradient_ah)*1e2:.4f} G/cm')

R_comp_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_comp_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_comp_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_comp_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
z_max_comp_slider = w.FloatSlider(description='z range (cm)', value=30.0, min=5.0, max=100.0, step=1.0)

@w.interact(R=R_comp_slider, I=I_comp_slider, d=d_comp_slider, N=N_comp_slider, z_max=z_max_comp_slider)
def update_comparison(R, I, d, N, z_max):
    plot_comparison(R/100, I, d/100, N, z_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## 2D Magnetic Field Visualization

Visualizing the field in the radial-axial plane.

In [ ]:
def B_anti_helmholtz_2d(R, I, d, r, z, N=1):
    R_grid, Z_grid = np.meshgrid(r, z)
    z1 = Z_grid + d/2
    rho1_sq = R_grid**2 + z1**2
    Bz1 = (mu_0 * I * N * R**2) / (2 * (R**2 + rho1_sq)**(3/2))
    Br1 = (mu_0 * I * N * R * R_grid * z1) / (2 * (R**2 + rho1_sq)**(5/2))
    z2 = Z_grid - d/2
    rho2_sq = R_grid**2 + z2**2
    Bz2 = -(mu_0 * I * N * R**2) / (2 * (R**2 + rho2_sq)**(3/2))
    Br2 = -(mu_0 * I * N * R * R_grid * z2) / (2 * (R**2 + rho2_sq)**(5/2))
    Bz = Bz1 + Bz2
    Br = Br1 + Br2
    B_total = np.sqrt(Bz**2 + Br**2)
    return Br, Bz, B_total

def plot_2d_field(R=0.1, I=10.0, d=0.1, N=10, r_max=0.15, z_max=0.15, n_points=100):
    r = np.linspace(-r_max, r_max, n_points)
    z = np.linspace(-z_max, z_max, n_points)
    Br, Bz, B_total = B_anti_helmholtz_2d(R, I, d, r, z, N)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    R_mesh, Z_mesh = np.meshgrid(r*100, z*100)
    
    contour1 = ax1.contourf(R_mesh, Z_mesh, Bz*1e4, levels=50, cmap=cmap_field)
    ax1.contour(R_mesh, Z_mesh, Bz*1e4, levels=20, colors='black', alpha=0.3)
    ax1.set_xlabel('r (cm)')
    ax1.set_ylabel('z (cm)')
    ax1.set_title('Bz component (G)')
    plt.colorbar(contour1, ax=ax1)
    ax1.axhline(0, color='gray', lw=0.5)
    ax1.axvline(0, color='gray', lw=0.5)
    
    contour2 = ax2.contourf(R_mesh, Z_mesh, B_total*1e4, levels=50, cmap='viridis')
    ax2.contour(R_mesh, Z_mesh, B_total*1e4, levels=20, colors='black', alpha=0.3)
    ax2.set_xlabel('r (cm)')
    ax2.set_ylabel('z (cm)')
    ax2.set_title('|B| magnitude (G)')
    plt.colorbar(contour2, ax=ax2)
    ax2.axhline(0, color='gray', lw=0.5)
    ax2.axvline(0, color='gray', lw=0.5)
    
    r_sparse = np.linspace(-r_max, r_max, 40)
    z_sparse = np.linspace(-z_max, z_max, 40)
    R_sparse, Z_sparse = np.meshgrid(r_sparse, z_sparse)
    Br_sparse, Bz_sparse, _ = B_anti_helmholtz_2d(R, I, d, r_sparse, z_sparse, N)
    ax3.streamplot(R_sparse*100, Z_sparse*100, Br_sparse*1e4, Bz_sparse*1e4, 
                  density=2, color='black', linewidth=0.8)
    ax3.set_xlabel('r (cm)')
    ax3.set_ylabel('z (cm)')
    ax3.set_title('Field lines')
    ax3.axhline(0, color='gray', lw=0.5)
    ax3.axvline(0, color='gray', lw=0.5)
    ax3.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

R_2d_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_2d_slider = w.FloatSlider(description='Current (A)', value=10.0, min=1.0, max=50.0, step=0.5)
d_2d_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_2d_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
r_max_2d_slider = w.FloatSlider(description='Range (cm)', value=15.0, min=1.0, max=50.0, step=1.0)

@w.interact(R=R_2d_slider, I=I_2d_slider, d=d_2d_slider, N=N_2d_slider, r_max=r_max_2d_slider)
def update_2d_field(R, I, d, N, r_max):
    plot_2d_field(R/100, I, d/100, N, r_max/100, r_max/100)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

## Application to Magneto-Optical Traps (MOTs)

Anti-Helmholtz coils are most commonly used in Magneto-Optical Traps for cooling and trapping neutral atoms. Here we calculate relevant parameters for MOT applications.

In [ ]:
species_data = {
    'Ca-40':   {'mass': 40.078,  'wavelength': 422.67276e-9, 'gamma': 2 * math.pi * 34.6e6},
    'Sr-88':   {'mass': 87.62,   'wavelength': 460.862e-9,    'gamma': 2 * math.pi * 32.0e6},
    'Rb-87':   {'mass': 86.909,  'wavelength': 780.241e-9,    'gamma': 2 * math.pi * 6.065e6},
    'Cs-133':  {'mass': 132.905, 'wavelength': 852.347e-9,    'gamma': 2 * math.pi * 5.234e6},
    'Na-23':   {'mass': 22.990,  'wavelength': 589.158e-9,    'gamma': 2 * math.pi * 9.79e6},
    'Yb-174':  {'mass': 173.045, 'wavelength': 398.911e-9,    'gamma': 2 * math.pi * 29.0e6},
}

def print_mot_params(R, I, d, N, species):
    data = species_data[species]
    mass = data['mass'] * amu
    wavelength = data['wavelength']
    gamma_val = data['gamma']
    
    gradient = compute_gradient_at_center(R, I, d, N)
    k = 2 * math.pi / wavelength
    v_recoil = hbar * k / mass
    # F_max = hbar * k * gamma_val / 2.0
    # a_max = F_max / mass
    T_Doppler = (hbar * gamma_val) / (2 * constants.k)
    
    print('=' * 60)
    print('MOT PARAMETERS')
    print('=' * 60)
    print(f'Configuration: R={R*100:.1f} cm, d={d*100:.1f} cm, I={I:.1f} A, N={N}')
    print(f'Species: {species}')
    print(f'\nField: Gradient = {abs(gradient)*1e2:.4f} G/cm')
    print(f'B at 1cm = {abs(gradient)*0.01*1e4:.2f} G')
    print(f'\nMOT: v_recoil = {v_recoil*100:.4f} cm/s') # a_max = {a_max:.2f} m/s^2')
    print(f'T_Doppler = {T_Doppler*1e6:.2f} microK')
    mass_amu = data['mass']
    print(f'\nSpecies: mass={mass_amu:.1f} amu, lambda={wavelength*1e9:.1f} nm')
    print('=' * 60)

R_mot_slider = w.FloatSlider(description='Radius (cm)', value=10.0, min=1.0, max=50.0, step=0.5)
I_mot_slider = w.FloatSlider(description='Current (A)', value=5.0, min=0.5, max=50.0, step=0.5)
d_mot_slider = w.FloatSlider(description='Separation (cm)', value=10.0, min=1.0, max=100.0, step=0.5)
N_mot_slider = w.IntSlider(description='Turns', value=10, min=1, max=100, step=1)
species_dropdown = w.Dropdown(description='Species', options=list(species_data.keys()), value='Ca-40')

mot_output = w.Output()

@w.interact(R=R_mot_slider, I=I_mot_slider, d=d_mot_slider, N=N_mot_slider, species=species_dropdown)
def update_mot(R, I, d, N, species):
    with mot_output:
        clear_output()
        print_mot_params(R/100, I, d/100, N, species)

display(mot_output)


interactive(children=(FloatSlider(value=10.0, description='Radius (cm)', max=50.0, min=1.0, step=0.5), FloatSl…

Output()

## Design Tool: Anti-Helmholtz Coils for Target Gradient

This tool helps design anti-Helmholtz coils to achieve a specific magnetic field gradient, which is often the starting point for MOT design.

In [10]:
def Bz_loop_on_axis(R, I, z, z_loop):
    """
    Magnetic field Bz on axis from a circular loop of radius R at position z_loop.
    R and z in meters, I in Amps. Returns Bz in Tesla.
    """
    z_rel = z - z_loop
    return (mu_0 * I * R**2) / (2 * (R**2 + z_rel**2)**(3/2))

def calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, config='antihelmholtz'):
    """
    Calculate Bz field along z-axis for given configuration.
    
    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current per coil in A
        z_points_mm: Array of z points where to calculate field in mm
        config: 'antihelmholtz' or 'helmholtz'
    
    Returns:
        Array of Bz values in Gauss
    """
    z_points_m = z_points_mm / 1000
    Bz = np.zeros_like(z_points_m)

    for R_mm in radii_mm:
        R_m = R_mm / 1000
        for z_layer_mm in z_positions_mm:
            z_layer_m = z_layer_mm / 1000

            if config == 'antihelmholtz':
                # Top: +I at +z_layer, Bottom: -I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz -= Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)
            else:  # helmholtz
                # Both: +I at +z_layer and +I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)

    return Bz * 1e4  # Convert T to Gauss

def calculate_gradient_in_zone(z_points_mm, Bz, zone_mm):
    """
    Calculate gradient (dB/dz) in the uniformity zone using linear regression.
    Returns gradient in G/mm.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    z_in_zone = z_points_mm[mask]
    Bz_in_zone = Bz[mask]

    if len(z_in_zone) < 2:
        return 0.0

    # Linear regression: B = slope * z + intercept
    # z is in mm, B is in G, so slope is in G/mm
    A = np.vstack([z_in_zone, np.ones(len(z_in_zone))]).T
    slope, _ = np.linalg.lstsq(A, Bz_in_zone, rcond=None)[0]

    # Convert from G/mm to G/cm (1 cm = 10 mm)
    return slope * 10

def calculate_uniformity_in_zone(Bz, zone_mm, z_points_mm):
    """
    Calculate field uniformity in zone as max percentage deviation from mean.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    Bz_in_zone = Bz[mask]

    if len(Bz_in_zone) == 0:
        return 0.0

    mean_B = np.mean(Bz_in_zone)
    if mean_B == 0:
        return 0.0

    max_deviation = np.max(np.abs(Bz_in_zone - mean_B)) / abs(mean_B) * 100
    return max_deviation

def plot_field_comparison(radii_mm, z_positions_mm, current_A, z_range_mm=(-50, 50), uniformity_zone_mm=5.0):
    """
    Plot magnetic field comparison with uniformity zone analysis.
    """
    z_points_mm = np.linspace(z_range_mm[0], z_range_mm[1], 500)

    # Calculate fields
    Bz_anti = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'antihelmholtz')
    Bz_helm = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'helmholtz')

    # Calculate metrics in the uniformity zone
    anti_gradient = calculate_gradient_in_zone(z_points_mm, Bz_anti, uniformity_zone_mm)
    helm_uniformity = calculate_uniformity_in_zone(Bz_helm, uniformity_zone_mm, z_points_mm)

    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Anti-Helmholtz plot
    ax1.plot(z_points_mm, Bz_anti, 'b-', lw=2, label='Anti-Helmholtz')
    ax1.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.fill_betweenx([min(Bz_anti), max(Bz_anti)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax1.axhline(0, color='black', lw=0.5)
    ax1.axvline(0, color='black', lw=0.5)
    ax1.set_xlabel('z (mm)')
    ax1.set_ylabel('Bz (G)')
    ax1.set_title(f'Anti-Helmholtz: Gradient = {anti_gradient:.2f} G/cm')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Helmholtz plot
    ax2.plot(z_points_mm, Bz_helm, 'r-', lw=2, label='Helmholtz')
    ax2.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.fill_betweenx([min(Bz_helm), max(Bz_helm)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax2.axhline(0, color='black', lw=0.5)
    ax2.axvline(0, color='black', lw=0.5)
    ax2.set_xlabel('z (mm)')
    ax2.set_ylabel('Bz (G)')
    ax2.set_title(f'Helmholtz: Uniformity = {helm_uniformity:.4f}% max deviation')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return anti_gradient, helm_uniformity

# Copper resistivity at 20°C in ohm-meter
RHO_CU = 1.72e-8

def calculate_electrical(radii_mm, z_positions_mm, current_A, wire_section_mm2, resistivity=RHO_CU):
    """
    Calculate electrical properties for the coil system.

    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current through each coil in A
        wire_section_mm2: Cross-sectional area of wire in mm²
        resistivity: Wire resistivity in ohm-meter (default: copper)

    Returns:
        Dictionary with electrical properties
    """
    # Convert to SI units
    radii_m = radii_mm / 1000
    wire_section_m2 = wire_section_mm2 / 1e6  # mm² to m²

    # Number of coils per layer and total coils
    num_coils_per_layer = len(radii_mm)
    num_layers = len(z_positions_mm)
    total_coils = num_coils_per_layer * num_layers

    # Total wire length in meters
    # Each coil is a circular turn: circumference = 2 * pi * R
    # Each radius appears once per layer
    total_length_m = 0.0
    for R in radii_m:
        circumference = 2 * np.pi * R
        total_length_m += circumference * num_layers
    top_bottom__coils_length = 2 * total_length_m  # For both top and bottom coils

    # Weight of copper wire (kg)
    density_cu = 8960  # kg/m³
    weight_kg_coil = density_cu * total_length_m * wire_section_m2
    top_bottom_weight_kg = 2 * weight_kg_coil  # For both top and bottom coils

    # Total resistance (ohms)
    resistance = resistivity * (total_length_m / wire_section_m2)
    top_bottom_resistance = 2 * resistance  # For both top and bottom coils

    # Voltage needed (V) - assuming coils are in series
    voltage = current_A * resistance
    top_bottom_voltage = 2 * voltage  # For both top and bottom coils

    # Power dissipated (W)
    power = voltage * current_A  # P = V * I = I² * R
    top_bottom_power = 2 * power  # For both top and bottom coils

    # Also calculate for parallel configuration
    resistance_parallel = resistivity * (total_length_m / wire_section_m2) / (total_coils ** 2)
    voltage_parallel = current_A * resistance_parallel * total_coils
    power_parallel = voltage_parallel * (current_A * total_coils)

    return {
        'total_length_m': total_length_m,
        'total_length_mm': total_length_m * 1000,
        'num_coils': total_coils,
        'resistance_series_ohm': resistance,
        'voltage_series_V': voltage,
        'power_series_W': power,
        'resistance_parallel_ohm': resistance_parallel,
        'voltage_parallel_V': voltage_parallel,
        'power_parallel_W': power_parallel,
        'wire_section_mm2': wire_section_mm2,
        'weight_kg': weight_kg_coil,
        'resistance_top_bottom_ohm': top_bottom_resistance,
        'voltage_top_bottom_V': top_bottom_voltage,
        'power_top_bottom_W': top_bottom_power,
        'weight_top_bottom_kg': top_bottom_weight_kg,
        'top_bottom_coils_length_m': top_bottom__coils_length
    }


def dB_dz_pair(R, I, a, N=1):
    """Compute the gradient of the magnetic field from a current loop at position 'a' equivalent to z.
    Generated by a pair of anti-Helmholtz coils (top and bottom) with current +I and -I respectively.
    Considering the quadrupole approximation for small z compared to R. The magnetic field is
    Btotal = B' (x, y, 2z)
    R: radius of the coil (m)
    I: current through the coil (A)
    a: axial position (m)
    N: number of turns
    Returns: dB/dz in Tesla/m"""
    denominator = (R**2 + a**2)**(5/2)
    dBz_dz = (3 * mu_0 * I * N * R**2 * a) / (2 * denominator)
    return dBz_dz

def generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um=5):
    """
    Generate concentric coil radii from min_r to max_r with spacing = thickness.
    Each coil is a ring at radius R with radial thickness. Plus, spacing_um is
    added to thickness for coil spacing.
    """
    spacing_mm = spacing_um / 1000  # Convert micrometers to mm
    effective_thickness = thickness_mm + spacing_mm
    radii = []
    current_r = min_r_mm + effective_thickness / 2  # Start at min_r + half thickness

    while current_r + effective_thickness / 2 <= max_r_mm:
        radii.append(current_r)
        current_r += effective_thickness  # Move to next position

    return np.array(radii)

def calculate_gradient_per_layer(radii_mm, current_A, z_position_mm):
    """
    Calculate total gradient from one layer of concentric coils.
    Each coil contributes based on its radius and z-position.
    """
    total_gradient = 0.0

    for R_mm in radii_mm:
        R_m = R_mm / 1000
        z_m = z_position_mm / 1000
        # Gradient from a coil pair (top + bottom)
        # For anti-Helmholtz: top coil at +z with +I, bottom at -z with -I
        grad = dB_dz_pair(R_m, current_A, z_m)  # From top and bottom coil (pair of anti-Helmholtz coils)
        total_gradient += grad

    # Convert from T/m to G/cm
    total_gradient_gcm = total_gradient * 100
    return total_gradient_gcm

def find_min_layers(target_gradient, min_r_mm, max_r_mm, thickness_mm,
                     current_A, min_z_mm=10.0, spacing_um=5):
    """
    Find minimum number of z-layers needed to achieve target gradient.
    
    Args:
        target_gradient: Target gradient in G/cm
        min_r_mm: Inner radius in mm
        max_r_mm: Outer radius in mm
        thickness_mm: Coil thickness in mm
        current_A: Current per coil in A
        min_z_mm: Minimum z position for first layer in mm
        spacing_um: Spacing between coils in micrometers
    Returns:
        num_layers, actual_gradient, radii, z_positions
    """
    # Generate concentric coil radii
    radii_mm = generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um)
    num_coils = len(radii_mm)

    if num_coils == 0:
        return 0, 0, [], []

    # Start with one layer
    layer_z_positions = []
    total_gradient = 0.0
    layer_count = 0

    # Add layers until we reach or exceed target
    while total_gradient < target_gradient:
        z_pos = min_z_mm + thickness_mm/2 + layer_count * thickness_mm  # Stack layers by thickness
        layer_z_positions.append(z_pos)
        
        # Gradient from this layer
        layer_grad = calculate_gradient_per_layer(radii_mm, current_A, z_pos)
        total_gradient += layer_grad
        layer_count += 1

        # Safety check
        if layer_count > 50:  # Prevent infinite loop
            break

    return layer_count, total_gradient, radii_mm, layer_z_positions

# Interactive widget to explore the configuration
# Create sliders as separate widgets (not inside @w.interact)
target_gradient_slider = w.FloatSlider(value=30.0, min=5.0, max=50.0, step=1.0,
                                      description='Target gradient (G/cm)', style={'description_width': 'initial'})
min_r_slider = w.FloatSlider(value=80.0, min=0.0, max=100.0, step=0.5,
                             description='Min radius (mm)', style={'description_width': 'initial'})
max_r_slider = w.FloatSlider(value=102.0, min=0.0, max=120.0, step=0.5,
                             description='Max radius (mm)', style={'description_width': 'initial'})
thickness_slider = w.FloatSlider(value=3.0, min=0.5, max=5.0, step=0.1,
                                description='Coil thickness (mm)', style={'description_width': 'initial'})
spacing_slider = w.FloatSlider(value=100.0, min=10.0, max=200.0, step=1.0,
                              description='Spacing (μm)', style={'description_width': 'initial'})
current_slider = w.FloatSlider(value=50.0, min=1.0, max=200.0, step=1.0,
                              description='Current per coil (A)', style={'description_width': 'initial'})
min_z_slider = w.FloatSlider(value=41.0, min=30.0, max=50.0, step=0.5,
                             description='Min z position (mm)', style={'description_width': 'initial'})
wire_section_slider = w.FloatSlider(value=2.6, min=0.1, max=10.0, step=0.1,
                                  description='Wire section (mm²)', style={'description_width': 'initial'})
uniformity_zone_slider = w.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                                     description='Uniformity zone (±mm)', style={'description_width': 'initial'})
layer_adjustment_slider = w.IntSlider(value=0, min=-3, max=3, step=1,
                                     description='Layer adjustment', style={'description_width': 'initial'})

# Create Run button and output
run_button = w.Button(description="Run Calculation", button_style='success')
layers_out = w.Output()

# Define the update function (without @w.interact)
def update_layers(target_gradient, min_r, max_r, thickness, spacing, current, min_z, wire_section, layer_adjustment):
    with layers_out:
        clear_output(wait=True)

        # Find optimal configuration first
        num_layers_optimal, total_grad_optimal, radii, z_positions_optimal  = find_min_layers(
            target_gradient, min_r, max_r, thickness, current, min_z, spacing
        )

        # Apply adjustment
        num_layers = max(1, num_layers_optimal + layer_adjustment)  # Ensure at least 1 layer

        # Generate adjusted z positions
        z_positions = []
        for i in range(num_layers):
            z_pos = min_z + thickness/2 + i * thickness
            z_positions.append(z_pos)
        z_positions = np.array(z_positions)

        # Calculate actual gradient with adjusted layers
        total_grad = 0.0
        for z_pos in z_positions:
            layer_grad = calculate_gradient_per_layer(radii, current, z_pos)
            total_grad += layer_grad

        # Electrical calculations
        elec = calculate_electrical(radii, z_positions, current, wire_section)

        print(f"Number of concentric coils per layer: {len(radii)}")
        print(f"Coil radii: {radii} mm")
        print(f"Number of z-layers: {num_layers}")
        print(f"Z-positions (wire center): {', '.join(f'{z:.0f}' for z in z_positions)} mm")
        print(f"Total coils: {elec['num_coils']}")
        print(f"Achieved gradient: {total_grad:.2f} G/cm")
        print("\n=== Electrical Properties (Copper Wire) ===")
        print(f"Wire cross-section: {elec['wire_section_mm2']:.2f} mm²")
        print(f"Copper resistivity: {RHO_CU:.2e} Ω·m")
        print(f"Copper density: 8960 kg/m³")
        print(f"\n--- Series Configuration 1 Coil---")
        print(f"Coil wire length: {elec['total_length_m']:.2f} m")
        print(f"Weight of wire per coil: {elec['weight_kg']:.2f} kg")
        print(f"Resistance: {elec['resistance_series_ohm']:.4f} Ω")
        print(f"Voltage needed: {elec['voltage_series_V']:.2f} V")
        print(f"Power dissipated: {elec['power_series_W']:.2f} W")
        print(f"\n--- Both Top and Bottom Coils ---")
        print(f"Top and bottom resistance: {elec['resistance_top_bottom_ohm']:.4f} Ω")
        print(f"Top and bottom voltage: {elec['voltage_top_bottom_V']:.2f} V")
        print(f"Top and bottom power: {elec['power_top_bottom_W']:.2f} W")
        print(f"Top and bottom coils length: {elec['top_bottom_coils_length_m']:.2f} m")
        print(f"Top and bottom weight: {elec['weight_top_bottom_kg']:.2f} kg")

        # Plot the configuration
        fig, ax = plt.subplots(figsize=(12, 8))

        # Draw boundaries
        # Setup limitations
        min_r_setup = 76  # mm
        max_r_setup = 100  # mm
        ax.axvline(x=min_r_setup, color='black', lw=2, alpha=0.7, label='Window Aperture')
        ax.axvline(x=-min_r_setup, color='black', lw=2, alpha=0.7)
        ax.axvline(x=max_r_setup, color='red', lw=2, linestyle=':', label='Side Windows')
        ax.axvline(x=-max_r_setup, color='red', lw=2, linestyle=':')
        ax.axhline(y=37.5, color='blue', lw=2, linestyle='--', label='Chamber Start')
        ax.axhline(y=66, color='purple', lw=2, linestyle=':', label='Screw Heads')

        # Reference lines
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)

        # Draw each coil in each layer
        colors = plt.cm.viridis(np.linspace(0, 1, num_layers))
        for layer_idx, z in enumerate(z_positions):
            color = colors[layer_idx]
            for R in radii:
                half_thick = thickness / 2
                for side in [-1, 1]:
                    x_coords = [side * (R - half_thick), side * (R + half_thick),
                               side * (R + half_thick), side * (R - half_thick)]
                    y_coords = [z - half_thick, z - half_thick,
                               z + half_thick, z + half_thick]
                    ax.fill(x_coords, y_coords, color=color, alpha=0.6,
                           edgecolor=color, lw=1)

        max_display_r = max_r + 5
        max_display_z = max(z_positions) + thickness + 5
        ax.set_xlim(-max_display_r, max_display_r)
        ax.set_ylim(0, max_display_z)
        ax.set_aspect('equal')
        ax.set_xlabel('x (mm)')
        ax.set_ylabel('z (mm)')
        ax.set_title(f'Concentric Coils: {num_layers} layers, {len(radii)} coils each')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Magnetic field plots with uniformity analysis
        uniformity_zone = uniformity_zone_slider.value
        anti_gradient, helm_uniformity = plot_field_comparison(
            radii, z_positions, current, z_range_mm=(-50, 50), uniformity_zone_mm=uniformity_zone
        )

        print("\n=== Magnetic Field Analysis ===")
        print(f"Uniformity zone: ±{uniformity_zone} mm")
        print(f"Anti-Helmholtz gradient in zone: {anti_gradient:.2f} G/cm")
        print(f"Helmholtz field uniformity in zone: {helm_uniformity:.4f}% max deviation")


# Connect button click to calculation
def on_run_click(b):
    update_layers(
        target_gradient_slider.value,
        min_r_slider.value,
        max_r_slider.value,
        thickness_slider.value,
        spacing_slider.value,
        current_slider.value,
        min_z_slider.value,
        wire_section_slider.value,
        layer_adjustment_slider.value
    )

run_button.on_click(on_run_click)

# Display all widgets in a vertical layout
display(w.VBox([
    target_gradient_slider,
    min_r_slider,
    max_r_slider,
    thickness_slider,
    spacing_slider,
    current_slider,
    min_z_slider,
    uniformity_zone_slider,
    wire_section_slider,
    layer_adjustment_slider,
    run_button,
    layers_out
]))

# Second Configuration: Above screw heads

In [11]:
def Bz_loop_on_axis(R, I, z, z_loop):
    """
    Magnetic field Bz on axis from a circular loop of radius R at position z_loop.
    R and z in meters, I in Amps. Returns Bz in Tesla.
    """
    z_rel = z - z_loop
    return (mu_0 * I * R**2) / (2 * (R**2 + z_rel**2)**(3/2))

def calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, config='antihelmholtz'):
    """
    Calculate Bz field along z-axis for given configuration.
    
    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current per coil in A
        z_points_mm: Array of z points where to calculate field in mm
        config: 'antihelmholtz' or 'helmholtz'
    
    Returns:
        Array of Bz values in Gauss
    """
    z_points_m = z_points_mm / 1000
    Bz = np.zeros_like(z_points_m)

    for R_mm in radii_mm:
        R_m = R_mm / 1000
        for z_layer_mm in z_positions_mm:
            z_layer_m = z_layer_mm / 1000

            if config == 'antihelmholtz':
                # Top: +I at +z_layer, Bottom: -I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz -= Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)
            else:  # helmholtz
                # Both: +I at +z_layer and +I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)

    return Bz * 1e4  # Convert T to Gauss

def calculate_gradient_in_zone(z_points_mm, Bz, zone_mm):
    """
    Calculate gradient (dB/dz) in the uniformity zone using linear regression.
    Returns gradient in G/mm.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    z_in_zone = z_points_mm[mask]
    Bz_in_zone = Bz[mask]

    if len(z_in_zone) < 2:
        return 0.0

    # Linear regression: B = slope * z + intercept
    # z is in mm, B is in G, so slope is in G/mm
    A = np.vstack([z_in_zone, np.ones(len(z_in_zone))]).T
    slope, _ = np.linalg.lstsq(A, Bz_in_zone, rcond=None)[0]

    # Convert from G/mm to G/cm (1 cm = 10 mm)
    return slope * 10

def calculate_uniformity_in_zone(Bz, zone_mm, z_points_mm):
    """
    Calculate field uniformity in zone as max percentage deviation from mean.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    Bz_in_zone = Bz[mask]

    if len(Bz_in_zone) == 0:
        return 0.0

    mean_B = np.mean(Bz_in_zone)
    if mean_B == 0:
        return 0.0

    max_deviation = np.max(np.abs(Bz_in_zone - mean_B)) / abs(mean_B) * 100
    return max_deviation

def plot_field_comparison(radii_mm, z_positions_mm, current_A, z_range_mm=(-50, 50), uniformity_zone_mm=5.0):
    """
    Plot magnetic field comparison with uniformity zone analysis.
    """
    z_points_mm = np.linspace(z_range_mm[0], z_range_mm[1], 500)

    # Calculate fields
    Bz_anti = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'antihelmholtz')
    Bz_helm = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'helmholtz')

    # Calculate metrics in the uniformity zone
    anti_gradient = calculate_gradient_in_zone(z_points_mm, Bz_anti, uniformity_zone_mm)
    helm_uniformity = calculate_uniformity_in_zone(Bz_helm, uniformity_zone_mm, z_points_mm)

    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Anti-Helmholtz plot
    ax1.plot(z_points_mm, Bz_anti, 'b-', lw=2, label='Anti-Helmholtz')
    ax1.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.fill_betweenx([min(Bz_anti), max(Bz_anti)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax1.axhline(0, color='black', lw=0.5)
    ax1.axvline(0, color='black', lw=0.5)
    ax1.set_xlabel('z (mm)')
    ax1.set_ylabel('Bz (G)')
    ax1.set_title(f'Anti-Helmholtz: Gradient = {anti_gradient:.2f} G/cm')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Helmholtz plot
    ax2.plot(z_points_mm, Bz_helm, 'r-', lw=2, label='Helmholtz')
    ax2.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.fill_betweenx([min(Bz_helm), max(Bz_helm)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax2.axhline(0, color='black', lw=0.5)
    ax2.axvline(0, color='black', lw=0.5)
    ax2.set_xlabel('z (mm)')
    ax2.set_ylabel('Bz (G)')
    ax2.set_title(f'Helmholtz: Uniformity = {helm_uniformity:.4f}% max deviation')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return anti_gradient, helm_uniformity

# Copper resistivity at 20°C in ohm-meter
RHO_CU = 1.72e-8

def calculate_electrical(radii_mm, z_positions_mm, current_A, wire_section_mm2, resistivity=RHO_CU):
    """
    Calculate electrical properties for the coil system.

    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current through each coil in A
        wire_section_mm2: Cross-sectional area of wire in mm²
        resistivity: Wire resistivity in ohm-meter (default: copper)

    Returns:
        Dictionary with electrical properties
    """
    # Convert to SI units
    radii_m = radii_mm / 1000
    wire_section_m2 = wire_section_mm2 / 1e6  # mm² to m²

    # Number of coils per layer and total coils
    num_coils_per_layer = len(radii_mm)
    num_layers = len(z_positions_mm)
    total_coils = num_coils_per_layer * num_layers

    # Total wire length in meters
    # Each coil is a circular turn: circumference = 2 * pi * R
    # Each radius appears once per layer
    total_length_m = 0.0
    for R in radii_m:
        circumference = 2 * np.pi * R
        total_length_m += circumference * num_layers
    top_bottom__coils_length = 2 * total_length_m  # For both top and bottom coils

    # Weight of copper wire (kg)
    density_cu = 8960  # kg/m³
    weight_kg_coil = density_cu * total_length_m * wire_section_m2
    top_bottom_weight_kg = 2 * weight_kg_coil  # For both top and bottom coils

    # Total resistance (ohms)
    resistance = resistivity * (total_length_m / wire_section_m2)
    top_bottom_resistance = 2 * resistance  # For both top and bottom coils

    # Voltage needed (V) - assuming coils are in series
    voltage = current_A * resistance
    top_bottom_voltage = 2 * voltage  # For both top and bottom coils

    # Power dissipated (W)
    power = voltage * current_A  # P = V * I = I² * R
    top_bottom_power = 2 * power  # For both top and bottom coils

    # Also calculate for parallel configuration
    resistance_parallel = resistivity * (total_length_m / wire_section_m2) / (total_coils ** 2)
    voltage_parallel = current_A * resistance_parallel * total_coils
    power_parallel = voltage_parallel * (current_A * total_coils)

    return {
        'total_length_m': total_length_m,
        'total_length_mm': total_length_m * 1000,
        'num_coils': total_coils,
        'resistance_series_ohm': resistance,
        'voltage_series_V': voltage,
        'power_series_W': power,
        'resistance_parallel_ohm': resistance_parallel,
        'voltage_parallel_V': voltage_parallel,
        'power_parallel_W': power_parallel,
        'wire_section_mm2': wire_section_mm2,
        'weight_kg': weight_kg_coil,
        'resistance_top_bottom_ohm': top_bottom_resistance,
        'voltage_top_bottom_V': top_bottom_voltage,
        'power_top_bottom_W': top_bottom_power,
        'weight_top_bottom_kg': top_bottom_weight_kg,
        'top_bottom_coils_length_m': top_bottom__coils_length
    }


def dB_dz_pair(R, I, a, N=1):
    """Compute the gradient of the magnetic field from a current loop at position 'a' equivalent to z.
    Generated by a pair of anti-Helmholtz coils (top and bottom) with current +I and -I respectively.
    Considering the quadrupole approximation for small z compared to R. The magnetic field is
    Btotal = B' (x, y, 2z)
    R: radius of the coil (m)
    I: current through the coil (A)
    a: axial position (m)
    N: number of turns
    Returns: dB/dz in Tesla/m"""
    denominator = (R**2 + a**2)**(5/2)
    dBz_dz = (3 * mu_0 * I * N * R**2 * a) / (2 * denominator)
    return dBz_dz

def generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um=5):
    """
    Generate concentric coil radii from min_r to max_r with spacing = thickness.
    Each coil is a ring at radius R with radial thickness. Plus, spacing_um is
    added to thickness for coil spacing.
    """
    spacing_mm = spacing_um / 1000  # Convert micrometers to mm
    effective_thickness = thickness_mm + spacing_mm
    radii = []
    current_r = min_r_mm + effective_thickness / 2  # Start at min_r + half thickness

    while current_r + effective_thickness / 2 <= max_r_mm:
        radii.append(current_r)
        current_r += effective_thickness  # Move to next position

    return np.array(radii)

def calculate_gradient_per_layer(radii_mm, current_A, z_position_mm):
    """
    Calculate total gradient from one layer of concentric coils.
    Each coil contributes based on its radius and z-position.
    """
    total_gradient = 0.0

    for R_mm in radii_mm:
        R_m = R_mm / 1000
        z_m = z_position_mm / 1000
        # Gradient from a coil pair (top + bottom)
        # For anti-Helmholtz: top coil at +z with +I, bottom at -z with -I
        grad = dB_dz_pair(R_m, current_A, z_m)  # From top and bottom coil (pair of anti-Helmholtz coils)
        total_gradient += grad

    # Convert from T/m to G/cm
    total_gradient_gcm = total_gradient * 100
    return total_gradient_gcm

def find_min_layers(target_gradient, min_r_mm, max_r_mm, thickness_mm,
                     current_A, min_z_mm=10.0, spacing_um=5):
    """
    Find minimum number of z-layers needed to achieve target gradient.
    
    Args:
        target_gradient: Target gradient in G/cm
        min_r_mm: Inner radius in mm
        max_r_mm: Outer radius in mm
        thickness_mm: Coil thickness in mm
        current_A: Current per coil in A
        min_z_mm: Minimum z position for first layer in mm
        spacing_um: Spacing between coils in micrometers
    Returns:
        num_layers, actual_gradient, radii, z_positions
    """
    # Generate concentric coil radii
    radii_mm = generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um)
    num_coils = len(radii_mm)

    if num_coils == 0:
        return 0, 0, [], []

    # Start with one layer
    layer_z_positions = []
    total_gradient = 0.0
    layer_count = 0

    # Add layers until we reach or exceed target
    while total_gradient < target_gradient:
        z_pos = min_z_mm + thickness_mm/2 + layer_count * thickness_mm  # Stack layers by thickness
        layer_z_positions.append(z_pos)
        
        # Gradient from this layer
        layer_grad = calculate_gradient_per_layer(radii_mm, current_A, z_pos)
        total_gradient += layer_grad
        layer_count += 1

        # Safety check
        if layer_count > 50:  # Prevent infinite loop
            break

    return layer_count, total_gradient, radii_mm, layer_z_positions

# Interactive widget to explore the configuration
# Create sliders as separate widgets (not inside @w.interact)
target_gradient_slider = w.FloatSlider(value=30.0, min=5.0, max=50.0, step=1.0,
                                      description='Target gradient (G/cm)', style={'description_width': 'initial'})
min_r_slider = w.FloatSlider(value=50.0, min=0.0, max=100.0, step=0.5,
                             description='Min radius (mm)', style={'description_width': 'initial'})
max_r_slider = w.FloatSlider(value=102.0, min=0.0, max=120.0, step=0.5,
                             description='Max radius (mm)', style={'description_width': 'initial'})
thickness_slider = w.FloatSlider(value=3.0, min=0.5, max=5.0, step=0.1,
                                description='Coil thickness (mm)', style={'description_width': 'initial'})
spacing_slider = w.FloatSlider(value=100.0, min=10.0, max=200.0, step=1.0,
                              description='Spacing (μm)', style={'description_width': 'initial'})
current_slider = w.FloatSlider(value=50.0, min=1.0, max=200.0, step=1.0,
                              description='Current per coil (A)', style={'description_width': 'initial'})
min_z_slider = w.FloatSlider(value=70.0, min=30.0, max=150.0, step=0.5,
                             description='Min z position (mm)', style={'description_width': 'initial'})
wire_section_slider = w.FloatSlider(value=2.6, min=0.1, max=10.0, step=0.1,
                                  description='Wire section (mm²)', style={'description_width': 'initial'})
uniformity_zone_slider = w.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                                     description='Uniformity zone (±mm)', style={'description_width': 'initial'})
layer_adjustment_slider = w.IntSlider(value=0, min=-3, max=3, step=1,
                                     description='Layer adjustment', style={'description_width': 'initial'})

# Create Run button and output
run_button = w.Button(description="Run Calculation", button_style='success')
layers_out = w.Output()

# Define the update function (without @w.interact)
def update_layers(target_gradient, min_r, max_r, thickness, spacing, current, min_z, wire_section, layer_adjustment):
    with layers_out:
        clear_output(wait=True)

        # Find optimal configuration first
        num_layers_optimal, total_grad_optimal, radii, z_positions_optimal  = find_min_layers(
            target_gradient, min_r, max_r, thickness, current, min_z, spacing
        )

        # Apply adjustment
        num_layers = max(1, num_layers_optimal + layer_adjustment)  # Ensure at least 1 layer

        # Generate adjusted z positions
        z_positions = []
        for i in range(num_layers):
            z_pos = min_z + thickness/2 + i * thickness
            z_positions.append(z_pos)
        z_positions = np.array(z_positions)

        # Calculate actual gradient with adjusted layers
        total_grad = 0.0
        for z_pos in z_positions:
            layer_grad = calculate_gradient_per_layer(radii, current, z_pos)
            total_grad += layer_grad

        # Electrical calculations
        elec = calculate_electrical(radii, z_positions, current, wire_section)

        print(f"Number of concentric coils per layer: {len(radii)}")
        print(f"Coil radii: {radii} mm")
        print(f"Number of z-layers: {num_layers}")
        print(f"Z-positions (wire center): {', '.join(f'{z:.0f}' for z in z_positions)} mm")
        print(f"Total coils: {elec['num_coils']}")
        print(f"Achieved gradient: {total_grad:.2f} G/cm")
        print("\n=== Electrical Properties (Copper Wire) ===")
        print(f"Wire cross-section: {elec['wire_section_mm2']:.2f} mm²")
        print(f"Copper resistivity: {RHO_CU:.2e} Ω·m")
        print(f"Copper density: 8960 kg/m³")
        print(f"\n--- Series Configuration 1 Coil---")
        print(f"Coil wire length: {elec['total_length_m']:.2f} m")
        print(f"Weight of wire per coil: {elec['weight_kg']:.2f} kg")
        print(f"Resistance: {elec['resistance_series_ohm']:.4f} Ω")
        print(f"Voltage needed: {elec['voltage_series_V']:.2f} V")
        print(f"Power dissipated: {elec['power_series_W']:.2f} W")
        print(f"\n--- Both Top and Bottom Coils ---")
        print(f"Top and bottom resistance: {elec['resistance_top_bottom_ohm']:.4f} Ω")
        print(f"Top and bottom voltage: {elec['voltage_top_bottom_V']:.2f} V")
        print(f"Top and bottom power: {elec['power_top_bottom_W']:.2f} W")
        print(f"Top and bottom coils length: {elec['top_bottom_coils_length_m']:.2f} m")
        print(f"Top and bottom weight: {elec['weight_top_bottom_kg']:.2f} kg")

        # Plot the configuration
        fig, ax = plt.subplots(figsize=(12, 8))

        # Draw boundaries
        # Setup limitations
        min_r_inner = 50  # mm
        min_r_setup = 76  # mm
        max_r_setup = 100  # mm
        ax.axvline(x=min_r_setup, color='black', lw=2, alpha=0.7, label='Window Aperture')
        ax.axvline(x=-min_r_setup, color='black', lw=2, alpha=0.7)
        ax.axvline(x=min_r_inner, color='black', lw=2, alpha=0.7, label='Window Inner Edge')
        ax.axvline(x=-min_r_inner, color='black', lw=2, alpha=0.7)
        ax.axvline(x=max_r_setup, color='red', lw=2, linestyle=':', label='Side Windows')
        ax.axvline(x=-max_r_setup, color='red', lw=2, linestyle=':')
        ax.axhline(y=37.5, color='blue', lw=2, linestyle='--', label='Chamber Start')
        ax.axhline(y=66, color='purple', lw=2, linestyle=':', label='Screw Heads')

        # Reference lines
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)

        # Draw each coil in each layer
        colors = plt.cm.viridis(np.linspace(0, 1, num_layers))
        for layer_idx, z in enumerate(z_positions):
            color = colors[layer_idx]
            for R in radii:
                half_thick = thickness / 2
                for side in [-1, 1]:
                    x_coords = [side * (R - half_thick), side * (R + half_thick),
                               side * (R + half_thick), side * (R - half_thick)]
                    y_coords = [z - half_thick, z - half_thick,
                               z + half_thick, z + half_thick]
                    ax.fill(x_coords, y_coords, color=color, alpha=0.6,
                           edgecolor=color, lw=1)

        max_display_r = max_r + 5
        max_display_z = max(z_positions) + thickness + 5
        ax.set_xlim(-max_display_r, max_display_r)
        ax.set_ylim(0, max_display_z)
        ax.set_aspect('equal')
        ax.set_xlabel('x (mm)')
        ax.set_ylabel('z (mm)')
        ax.set_title(f'Concentric Coils: {num_layers} layers, {len(radii)} coils each')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Magnetic field plots with uniformity analysis
        uniformity_zone = uniformity_zone_slider.value
        anti_gradient, helm_uniformity = plot_field_comparison(
            radii, z_positions, current, z_range_mm=(-50, 50), uniformity_zone_mm=uniformity_zone
        )

        print("\n=== Magnetic Field Analysis ===")
        print(f"Uniformity zone: ±{uniformity_zone} mm")
        print(f"Anti-Helmholtz gradient in zone: {anti_gradient:.2f} G/cm")
        print(f"Helmholtz field uniformity in zone: {helm_uniformity:.4f}% max deviation")


# Connect button click to calculation
def on_run_click(b):
    update_layers(
        target_gradient_slider.value,
        min_r_slider.value,
        max_r_slider.value,
        thickness_slider.value,
        spacing_slider.value,
        current_slider.value,
        min_z_slider.value,
        wire_section_slider.value,
        layer_adjustment_slider.value
    )

run_button.on_click(on_run_click)

# Display all widgets in a vertical layout
display(w.VBox([
    target_gradient_slider,
    min_r_slider,
    max_r_slider,
    thickness_slider,
    spacing_slider,
    current_slider,
    min_z_slider,
    uniformity_zone_slider,
    wire_section_slider,
    layer_adjustment_slider,
    run_button,
    layers_out
]))

# Third optional configuration: Combiniation of both above and below screw heads

In [13]:
# Copper resistivity at 20°C in ohm-meter
RHO_CU = 1.72e-8

# Store the last computed geometry for intensity variation
last_geometry = None  # Stores (radii_outer, z_positions_outer, radii_inner, z_positions_inner, thickness, wire_section)


def update_intensity(change): 
    global last_geometry
    if last_geometry is None:
        with intensity_out:
            clear_output(wait=True)
            print("Please compute a configuration first by clicking 'Run Calculation'")
        return

    current = change.new
    radii_outer, z_positions_outer, radii_inner, z_positions_inner, thickness, wire_section = last_geometry

    with intensity_out:
        clear_output(wait=True)

        # Calculate gradient with new current
        total_grad = calculate_total_gradient(radii_outer, z_positions_outer, current)
        total_grad += calculate_total_gradient(radii_inner, z_positions_inner, current)

        # Electrical properties
        all_radii = []
        all_z = []
        for z in z_positions_outer:
            all_radii.extend(radii_outer.tolist())
            all_z.extend([z] * len(radii_outer))
        for z in z_positions_inner:
            all_radii.extend(radii_inner.tolist())
            all_z.extend([z] * len(radii_inner))

        elec = calculate_electrical(np.array(all_radii), np.array(all_z), current, wire_section)

        # Print summary
        num_outer = len(z_positions_outer)
        num_inner = len(z_positions_inner)
        all_z_for_plot = np.concatenate([z_positions_outer, z_positions_inner])
        all_radii_for_plot = np.concatenate([radii_outer, radii_inner])

        print(f"=== Intensity Variation (Fixed Geometry) ===")
        print(f"Current: {current:.1f} A")
        print(f"Configuration: {num_outer} outer layers + {num_inner} inner layers")
        print(f"Gradient: {total_grad:.2f} G/cm")
        print(f"\n=== Electrical Properties (Single Side) ===")
        print(f"Voltage needed: {elec['voltage_series_V']:.2f} V")
        print(f"Power dissipated: {elec['power_series_W']:.2f} W")
        print(f"Resistance: {elec['resistance_series_ohm']:.4f} Ω")
        print(f"Total wire length: {elec['total_length_m']:.2f} m")
        print(f"Total weight: {elec['weight_kg']:.2f} kg")
        print(f"\n=== Electrical Properties (Top + Bottom Coils) ===")
        print(f"Voltage needed: {elec['voltage_top_bottom_V']:.2f} V")
        print(f"Power dissipated: {elec['power_top_bottom_W']:.2f} W")
        print(f"Resistance: {elec['resistance_top_bottom_ohm']:.4f} Ω")
        print(f"Total wire length: {elec['top_bottom_coils_length_m']:.2f} m")
        print(f"Total weight: {elec['weight_top_bottom_kg']:.2f} kg")

        # Plot magnetic field
        uniformity_zone = uniformity_zone_slider.value
        z_points_mm = np.linspace(-50, 50, 500)

        # Calculate outer and inner contributions SEPARATELY to avoid cartesian product
        Bz_anti = np.zeros_like(z_points_mm)
        Bz_helm = np.zeros_like(z_points_mm)

        if len(radii_outer) > 0 and len(z_positions_outer) > 0:
            Bz_anti += calculate_field_on_axis(radii_outer, z_positions_outer, current, z_points_mm, 'antihelmholtz')
            Bz_helm += calculate_field_on_axis(radii_outer, z_positions_outer, current, z_points_mm, 'helmholtz')
        if len(radii_inner) > 0 and len(z_positions_inner) > 0:
            Bz_anti += calculate_field_on_axis(radii_inner, z_positions_inner, current, z_points_mm, 'antihelmholtz')
            Bz_helm += calculate_field_on_axis(radii_inner, z_positions_inner, current, z_points_mm, 'helmholtz')

        # Calculate metrics
        anti_gradient = calculate_gradient_in_zone(z_points_mm, Bz_anti, uniformity_zone)
        helm_uniformity = calculate_uniformity_in_zone(Bz_helm, uniformity_zone, z_points_mm)

        # Plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
        ax1.plot(z_points_mm, Bz_anti, 'b-', lw=2, label='Anti-Helmholtz')
        ax1.axvline(x=uniformity_zone, color='green', lw=1, linestyle='--')
        ax1.axvline(x=-uniformity_zone, color='green', lw=1, linestyle='--')
        ax1.fill_betweenx([min(Bz_anti), max(Bz_anti)], -uniformity_zone, uniformity_zone,
                        color='green', alpha=0.1, label=f'Zone (±{uniformity_zone}mm)')
        ax1.axhline(0, color='black', lw=0.5)
        ax1.axvline(0, color='black', lw=0.5)
        ax1.set_xlabel('z (mm)')
        ax1.set_ylabel('Bz (G)')
        ax1.set_title(f'Anti-Helmholtz: Gradient = {anti_gradient:.2f} G/cm')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        ax2.plot(z_points_mm, Bz_helm, 'r-', lw=2, label='Helmholtz')
        ax2.axvline(x=uniformity_zone, color='green', lw=1, linestyle='--')
        ax2.axvline(x=-uniformity_zone, color='green', lw=1, linestyle='--')
        ax2.fill_betweenx([min(Bz_helm), max(Bz_helm)], -uniformity_zone, uniformity_zone,
                        color='green', alpha=0.1, label=f'Zone (±{uniformity_zone}mm)')
        ax2.axhline(0, color='black', lw=0.5)
        ax2.axvline(0, color='black', lw=0.5)
        ax2.set_xlabel('z (mm)')
        ax2.set_ylabel('Bz (G)')
        ax2.set_title(f'Helmholtz: Uniformity = {helm_uniformity:.4f}% max deviation')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print(f"Anti-Helmholtz gradient in zone: {anti_gradient:.2f} G/cm")


def Bz_loop_on_axis(R, I, z, z_loop):
    """
    Magnetic field Bz on axis from a circular loop of radius R at position z_loop.
    R and z in meters, I in Amps. Returns Bz in Tesla.
    """
    z_rel = z - z_loop
    return (mu_0 * I * R**2) / (2 * (R**2 + z_rel**2)**(3/2))

def calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, config='antihelmholtz'):
    """
    Calculate Bz field along z-axis for given configuration.
    
    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current per coil in A
        z_points_mm: Array of z points where to calculate field in mm
        config: 'antihelmholtz' or 'helmholtz'
    
    Returns:
        Array of Bz values in Gauss
    """
    z_points_m = z_points_mm / 1000
    Bz = np.zeros_like(z_points_m)

    for R_mm in radii_mm:
        R_m = R_mm / 1000
        for z_layer_mm in z_positions_mm:
            z_layer_m = z_layer_mm / 1000

            if config == 'antihelmholtz':
                # Top: +I at +z_layer, Bottom: -I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz -= Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)
            else:  # helmholtz
                # Both: +I at +z_layer and +I at -z_layer
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, z_layer_m)
                Bz += Bz_loop_on_axis(R_m, current_A, z_points_m, -z_layer_m)

    return Bz * 1e4  # Convert T to Gauss

def calculate_gradient_in_zone(z_points_mm, Bz, zone_mm):
    """
    Calculate gradient (dB/dz) in the uniformity zone using linear regression.
    Returns gradient in G/mm.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    z_in_zone = z_points_mm[mask]
    Bz_in_zone = Bz[mask]

    if len(z_in_zone) < 2:
        return 0.0

    # Linear regression: B = slope * z + intercept
    # z is in mm, B is in G, so slope is in G/mm
    A = np.vstack([z_in_zone, np.ones(len(z_in_zone))]).T
    slope, _ = np.linalg.lstsq(A, Bz_in_zone, rcond=None)[0]

    # Convert from G/mm to G/cm (1 cm = 10 mm)
    return slope * 10

def calculate_uniformity_in_zone(Bz, zone_mm, z_points_mm):
    """
    Calculate field uniformity in zone as max percentage deviation from mean.
    """
    mask = (np.abs(z_points_mm) <= zone_mm)
    Bz_in_zone = Bz[mask]

    if len(Bz_in_zone) == 0:
        return 0.0

    mean_B = np.mean(Bz_in_zone)
    if mean_B == 0:
        return 0.0

    max_deviation = np.max(np.abs(Bz_in_zone - mean_B)) / abs(mean_B) * 100
    return max_deviation

def plot_field_comparison(radii_mm, z_positions_mm, current_A, z_range_mm=(-50, 50), uniformity_zone_mm=5.0):
    """
    Plot magnetic field comparison with uniformity zone analysis.
    """
    z_points_mm = np.linspace(z_range_mm[0], z_range_mm[1], 500)

    # Calculate fields
    Bz_anti = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'antihelmholtz')
    Bz_helm = calculate_field_on_axis(radii_mm, z_positions_mm, current_A, z_points_mm, 'helmholtz')

    # Calculate metrics in the uniformity zone
    anti_gradient = calculate_gradient_in_zone(z_points_mm, Bz_anti, uniformity_zone_mm)
    helm_uniformity = calculate_uniformity_in_zone(Bz_helm, uniformity_zone_mm, z_points_mm)

    # Create figure
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

    # Anti-Helmholtz plot
    ax1.plot(z_points_mm, Bz_anti, 'b-', lw=2, label='Anti-Helmholtz')
    ax1.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax1.fill_betweenx([min(Bz_anti), max(Bz_anti)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax1.axhline(0, color='black', lw=0.5)
    ax1.axvline(0, color='black', lw=0.5)
    ax1.set_xlabel('z (mm)')
    ax1.set_ylabel('Bz (G)')
    ax1.set_title(f'Anti-Helmholtz: Gradient = {anti_gradient:.2f} G/cm')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Helmholtz plot
    ax2.plot(z_points_mm, Bz_helm, 'r-', lw=2, label='Helmholtz')
    ax2.axvline(x=uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.axvline(x=-uniformity_zone_mm, color='green', lw=1, linestyle='--')
    ax2.fill_betweenx([min(Bz_helm), max(Bz_helm)], -uniformity_zone_mm, uniformity_zone_mm,
                     color='green', alpha=0.1, label=f'Zone (±{uniformity_zone_mm}mm)')
    ax2.axhline(0, color='black', lw=0.5)
    ax2.axvline(0, color='black', lw=0.5)
    ax2.set_xlabel('z (mm)')
    ax2.set_ylabel('Bz (G)')
    ax2.set_title(f'Helmholtz: Uniformity = {helm_uniformity:.4f}% max deviation')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return anti_gradient, helm_uniformity


def calculate_electrical(radii_mm, z_positions_mm, current_A, wire_section_mm2, radii_inner_mm, support_mm, z_screw_heads_mm, resistivity=RHO_CU):
    """
    Calculate electrical properties for the coil system.

    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (one per layer) in mm
        current_A: Current through each coil in A
        wire_section_mm2: Cross-sectional area of wire in mm²
        resistivity: Wire resistivity in ohm-meter (default: copper)

    Returns:
        Dictionary with electrical properties
    """
    # Convert to SI units
    radii_m = radii_mm / 1000
    wire_section_m2 = wire_section_mm2 / 1e6  # mm² to m²

    # Number of coils per layer and total coils
    num_coils_per_layer = len(radii_mm)
    num_layers = len(z_positions_mm)
    total_coils = num_coils_per_layer * num_layers

    # Total wire length in meters
    # Each coil is a circular turn: circumference = 2 * pi * R
    # Each radius appears once per layer
    total_length_m = 0.0
    for R in radii_m:
        circumference = 2 * np.pi * R
        total_length_m += circumference * num_layers
    top_bottom__coils_length = 2 * total_length_m  # For both top and bottom coils

    # Weight of copper wire (kg)
    density_cu = 8960  # kg/m³
    weight_kg_coil = density_cu * total_length_m * wire_section_m2
    top_bottom_weight_kg = 2 * weight_kg_coil  # For both top and bottom coils

    # Total resistance (ohms)
    resistance = resistivity * (total_length_m / wire_section_m2)
    top_bottom_resistance = 2 * resistance  # For both top and bottom coils

    # Voltage needed (V) - assuming coils are in series
    voltage = current_A * resistance
    top_bottom_voltage = 2 * voltage  # For both top and bottom coils

    # Power dissipated (W)
    power = voltage * current_A  # P = V * I = I² * R
    top_bottom_power = 2 * power  # For both top and bottom coils

    # Also calculate for parallel configuration
    resistance_parallel = resistivity * (total_length_m / wire_section_m2) / (total_coils ** 2)
    voltage_parallel = current_A * resistance_parallel * total_coils
    power_parallel = voltage_parallel * (current_A * total_coils)

    return {
        'total_length_m': total_length_m,
        'total_length_mm': total_length_m * 1000,
        'num_coils': total_coils,
        'resistance_series_ohm': resistance,
        'voltage_series_V': voltage,
        'power_series_W': power,
        'resistance_parallel_ohm': resistance_parallel,
        'voltage_parallel_V': voltage_parallel,
        'power_parallel_W': power_parallel,
        'wire_section_mm2': wire_section_mm2,
        'weight_kg': weight_kg_coil,
        'resistance_top_bottom_ohm': top_bottom_resistance,
        'voltage_top_bottom_V': top_bottom_voltage,
        'power_top_bottom_W': top_bottom_power,
        'weight_top_bottom_kg': top_bottom_weight_kg,
        'top_bottom_coils_length_m': top_bottom__coils_length
    }


def dB_dz_pair(R, I, a, N=1):
    """Compute the gradient of the magnetic field from a current loop at position 'a' equivalent to z.
    Generated by a pair of anti-Helmholtz coils (top and bottom) with current +I and -I respectively.
    Considering the quadrupole approximation for small z compared to R. The magnetic field is
    Btotal = B' (x, y, 2z)
    R: radius of the coil (m)
    I: current through the coil (A)
    a: axial position (m)
    N: number of turns
    Returns: dB/dz in Tesla/m"""
    denominator = (R**2 + a**2)**(5/2)
    dBz_dz = (3 * mu_0 * I * N * R**2 * a) / (2 * denominator)
    return dBz_dz

def generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um=5):
    """
    Generate concentric coil radii from min_r to max_r with spacing = thickness.
    Each coil is a ring at radius R with radial thickness. Plus, spacing_um is
    added to thickness for coil spacing.
    """
    spacing_mm = spacing_um / 1000  # Convert micrometers to mm
    effective_thickness = thickness_mm + spacing_mm
    radii = []
    current_r = min_r_mm + effective_thickness / 2  # Start at min_r + half thickness

    while current_r + effective_thickness / 2 <= max_r_mm:
        radii.append(current_r)
        current_r += effective_thickness  # Move to next position

    return np.array(radii)


def calculate_total_gradient(radii_mm, z_positions_mm, current_A):
    """
    Calculate total gradient from all coils.
    
    Args:
        radii_mm: Array of coil radii in mm
        z_positions_mm: Array of z positions (each z is a layer, containing all radii)
        current_A: Current per coil in A
    
    Returns:
        Total gradient in G/cm
    """
    total_gradient = 0.0
    for z_pos_mm in z_positions_mm:
        for R_mm in radii_mm:
            R_m = R_mm / 1000
            z_m = z_pos_mm / 1000
            total_gradient += dB_dz_pair(R_m, current_A, z_m)
    return total_gradient * 100  # Convert T/m to G/cm

def find_min_layers(target_gradient, min_r_mm, max_r_mm, thickness_mm,
                     current_A, min_z_mm=10.0, spacing_um=5, z_screw_heads_mm=66.0,
                     min_r_inner_mm=50.0, support_mm=3.0):
    """
    Find minimum number of z-layers needed to achieve target gradient.
    
    Returns:
        tuple: (num_layers_outer, total_gradient, radii_outer, z_positions_outer,
                num_layers_inner, radii_inner, z_positions_inner)
    """
    # Generate concentric coil radii
    radii_outer = generate_concentric_coils(min_r_mm, max_r_mm, thickness_mm, spacing_um)
    radii_inner = generate_concentric_coils(min_r_inner_mm, max_r_mm, thickness_mm, spacing_um)
    
    # Handle empty cases
    if len(radii_outer) == 0:
        radii_outer = np.array([])
    if len(radii_inner) == 0:
        radii_inner = np.array([])
    
    # Start with outer layers below screw heads
    z_positions_outer = []
    total_gradient = 0.0
    layer_count = 0

    # Add outer layers until we reach screw heads or target
    while len(radii_outer) > 0 and (min_z_mm + (layer_count + 1) * thickness_mm) < z_screw_heads_mm:
        z_pos = min_z_mm + thickness_mm/2 + layer_count * thickness_mm
        z_positions_outer.append(z_pos)
        
        # Gradient from this layer (all outer coils at this z)
        layer_grad = calculate_total_gradient(radii_outer, np.array([z_pos]), current_A)
        total_gradient += layer_grad
        layer_count += 1

        # Safety checks
        if layer_count > 50 or total_gradient >= target_gradient:
            break

    # Add inner layers above screw heads if target not reached
    z_positions_inner = []
    layer_count_inner = 0

    if total_gradient < target_gradient and len(radii_inner) > 0:
        print("Warning: Could not reach target gradient before hitting screw heads. Adding inner coils.")
        
        while total_gradient < target_gradient:
            z_pos = z_screw_heads_mm + support_mm + thickness_mm/2 + layer_count_inner * thickness_mm
            z_positions_inner.append(z_pos)
            
            layer_grad = calculate_total_gradient(radii_inner, np.array([z_pos]), current_A)
            total_gradient += layer_grad
            layer_count_inner += 1

            if layer_count_inner > 50:
                print("Warning: Could not reach target gradient even with inner coils.")
                break

    return (layer_count, total_gradient, radii_outer, np.array(z_positions_outer),
            layer_count_inner, radii_inner, np.array(z_positions_inner))

def calculate_electrical(all_radii_mm, all_z_positions_mm, current_A, wire_section_mm2, resistivity=RHO_CU):
    """
    Calculate electrical properties for the coil system.
    
    Args:
        all_radii_mm: 1D array of ALL coil radii in mm (each radius repeated for each layer it appears in)
        all_z_positions_mm: 1D array of z positions in mm (same length as all_radii_mm)
        current_A: Current through each coil in A
        wire_section_mm2: Cross-sectional area of wire in mm²
        resistivity: Wire resistivity in ohm-meter
    
    Returns:
        Dictionary with electrical properties
    """
    if len(all_radii_mm) == 0:
        return {
            'total_length_m': 0, 'total_length_mm': 0, 'num_coils': 0,
            'resistance_series_ohm': 0, 'voltage_series_V': 0, 'power_series_W': 0,
            'resistance_parallel_ohm': 0, 'voltage_parallel_V': 0, 'power_parallel_W': 0,
            'wire_section_mm2': wire_section_mm2, 'weight_kg': 0,
            'resistance_top_bottom_ohm': 0, 'voltage_top_bottom_V': 0,
            'power_top_bottom_W': 0, 'weight_top_bottom_kg': 0,
            'top_bottom_coils_length_m': 0
        }
    
    radii_m = all_radii_mm / 1000
    wire_section_m2 = wire_section_mm2 / 1e6
    
    num_coils = len(all_radii_mm)
    
    # Total wire length: sum of circumferences for all coils
    total_length_m = np.sum(2 * np.pi * radii_m)
    top_bottom_coils_length_m = 2 * total_length_m
    
    # Weight of copper wire
    density_cu = 8960  # kg/m³
    weight_kg = density_cu * total_length_m * wire_section_m2
    top_bottom_weight_kg = 2 * weight_kg
    
    # Resistance (series)
    resistance = resistivity * (total_length_m / wire_section_m2)
    top_bottom_resistance = 2 * resistance
    
    # Voltage and power (series)
    voltage = current_A * resistance
    top_bottom_voltage = 2 * voltage
    power = voltage * current_A
    top_bottom_power = 2 * power
    
    # Parallel configuration
    if num_coils > 0:
        # For parallel: each coil has its own resistance
        # Total resistance: 1/R_total = sum(1/R_i) for all coils
        # Since all coils have same wire section and resistivity:
        # R_i = resistivity * (2*pi*R_i) / wire_section_m2
        coil_resistances = resistivity * (2 * np.pi * radii_m) / wire_section_m2
        resistance_parallel = 1.0 / np.sum(1.0 / coil_resistances) if num_coils > 0 else 0
        voltage_parallel = current_A * resistance_parallel
        power_parallel = voltage_parallel * current_A
    else:
        resistance_parallel = 0
        voltage_parallel = 0
        power_parallel = 0
    
    return {
        'total_length_m': total_length_m,
        'total_length_mm': total_length_m * 1000,
        'num_coils': num_coils,
        'resistance_series_ohm': resistance,
        'voltage_series_V': voltage,
        'power_series_W': power,
        'resistance_parallel_ohm': resistance_parallel,
        'voltage_parallel_V': voltage_parallel,
        'power_parallel_W': power_parallel,
        'wire_section_mm2': wire_section_mm2,
        'weight_kg': weight_kg,
        'resistance_top_bottom_ohm': top_bottom_resistance,
        'voltage_top_bottom_V': top_bottom_voltage,
        'power_top_bottom_W': top_bottom_power,
        'weight_top_bottom_kg': top_bottom_weight_kg,
        'top_bottom_coils_length_m': top_bottom_coils_length_m
    }

def update_layers(target_gradient, min_r, max_r, thickness, spacing, current, min_z, wire_section, layer_adjustment, z_screw_heads, r_min_inner, support):
    with layers_out:
        clear_output(wait=True)

        # Find optimal configuration
        (num_layers_outer, total_grad_optimal, radii_outer, z_positions_outer,
         num_layers_inner, radii_inner, z_positions_inner) = find_min_layers(
            target_gradient, min_r, max_r, thickness, current, min_z, spacing,
            z_screw_heads_mm=z_screw_heads, min_r_inner_mm=r_min_inner, support_mm=support
        )

        # Apply adjustment to INNER layers (above screw heads) - NOT outer layers
        num_layers_inner = max(0, num_layers_inner + layer_adjustment)
        z_positions_inner = np.array([
            z_screw_heads + support + thickness/2 + i * thickness
            for i in range(num_layers_inner)
        ])

        # Calculate actual gradient with adjusted layers
        total_grad = calculate_total_gradient(radii_outer, z_positions_outer, current)
        total_grad += calculate_total_gradient(radii_inner, z_positions_inner, current)

        # Create flat arrays for all coils (each radius with each z position in its region)
        all_radii = []
        all_z_positions = []
        
        # Outer coils: each outer radius appears at each outer z position
        for z in z_positions_outer:
            all_radii.extend(radii_outer.tolist())
            all_z_positions.extend([z] * len(radii_outer))
        
        # Inner coils: each inner radius appears at each inner z position
        for z in z_positions_inner:
            all_radii.extend(radii_inner.tolist())
            all_z_positions.extend([z] * len(radii_inner))
        
        all_radii = np.array(all_radii)
        all_z_positions = np.array(all_z_positions)

        # Electrical calculations
        elec = calculate_electrical(all_radii, all_z_positions, current, wire_section)

        # Print results
        print(f"Number of concentric coils per outer layer: {len(radii_outer)}")
        print(f"Number of concentric coils per inner layer: {len(radii_inner)}")
        print(f"Outer coil radii: {radii_outer} mm")
        print(f"Inner coil radii: {radii_inner} mm")
        print(f"Number of outer z-layers: {num_layers_outer}")
        print(f"Number of inner z-layers: {num_layers_inner}")
        print(f"Outer Z-positions (wire center): {', '.join(f'{z:.0f}' for z in z_positions_outer)} mm")
        print(f"Inner Z-positions (wire center): {', '.join(f'{z:.0f}' for z in z_positions_inner)} mm")
        print(f"Total coils: {elec['num_coils']}")
        print(f"Achieved gradient: {total_grad:.2f} G/cm")

        print("\n=== Electrical Properties (Copper Wire) ===")
        print(f"Wire cross-section: {elec['wire_section_mm2']:.2f} mm²")
        print(f"Copper resistivity: {RHO_CU:.2e} Ω·m")
        print(f"Copper density: 8960 kg/m³")
        print(f"\n--- Series Configuration ---")
        print(f"Total wire length: {elec['total_length_m']:.2f} m")
        print(f"Total weight: {elec['weight_kg']:.2f} kg")
        print(f"Resistance: {elec['resistance_series_ohm']:.4f} Ω")
        print(f"Voltage needed: {elec['voltage_series_V']:.2f} V")
        print(f"Power dissipated: {elec['power_series_W']:.2f} W")
        print(f"\n--- Both Top and Bottom Coils ---")
        print(f"Top+bottom wire length: {elec['top_bottom_coils_length_m']:.2f} m")
        print(f"Top+bottom resistance: {elec['resistance_top_bottom_ohm']:.4f} Ω")
        print(f"Top+bottom voltage: {elec['voltage_top_bottom_V']:.2f} V")
        print(f"Top+bottom power: {elec['power_top_bottom_W']:.2f} W")
        print(f"Top+bottom weight: {elec['weight_top_bottom_kg']:.2f} kg")

        # Plot the configuration
        all_z_for_plot = np.concatenate([z_positions_outer, z_positions_inner])
        num_layers = len(all_z_for_plot)

        fig, ax = plt.subplots(figsize=(12, 8))

        # Draw boundaries
        min_r_inner_plot = 50  # mm
        min_r_setup = 76  # mm
        max_r_setup = 100  # mm
        ax.axvline(x=min_r_setup, color='black', lw=2, alpha=0.7, label='Window Aperture')
        ax.axvline(x=-min_r_setup, color='black', lw=2, alpha=0.7)
        ax.axvline(x=min_r_inner_plot, color='black', lw=2, alpha=0.7, label='Window Inner Edge')
        ax.axvline(x=-min_r_inner_plot, color='black', lw=2, alpha=0.7)
        ax.axvline(x=max_r_setup, color='red', lw=2, linestyle=':', label='Side Windows')
        ax.axvline(x=-max_r_setup, color='red', lw=2, linestyle=':')
        ax.axhline(y=37.5, color='blue', lw=2, linestyle='--', label='Chamber Start')
        ax.axhline(y=66, color='purple', lw=2, linestyle=':', label='Screw Heads')

        # Reference lines
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)

        # Draw each coil in each layer
        colors = plt.cm.viridis(np.linspace(0, 1, num_layers))
        layer_idx = 0
        
        # Draw outer coils
        for z in z_positions_outer:
            color = colors[layer_idx % len(colors)]
            for R in radii_outer:
                half_thick = thickness / 2
                for side in [-1, 1]:
                    x_coords = [side * (R - half_thick), side * (R + half_thick),
                               side * (R + half_thick), side * (R - half_thick)]
                    y_coords = [z - half_thick, z - half_thick,
                               z + half_thick, z + half_thick]
                    ax.fill(x_coords, y_coords, color=color, alpha=0.6,
                           edgecolor=color, lw=1)
            layer_idx += 1

        # Draw inner coils
        for z in z_positions_inner:
            color = colors[layer_idx % len(colors)]
            for R in radii_inner:
                half_thick = thickness / 2
                for side in [-1, 1]:
                    x_coords = [side * (R - half_thick), side * (R + half_thick),
                               side * (R + half_thick), side * (R - half_thick)]
                    y_coords = [z - half_thick, z - half_thick,
                               z + half_thick, z + half_thick]
                    ax.fill(x_coords, y_coords, color=color, alpha=0.6,
                           edgecolor=color, lw=1)
            layer_idx += 1

        max_display_r = max(max_r, max(radii_inner) if len(radii_inner) > 0 else 0) + 5
        max_display_z = max(np.max(z_positions_outer) if len(z_positions_outer) > 0 else 0,
                            np.max(z_positions_inner) if len(z_positions_inner) > 0 else 0) + thickness + 5
        ax.set_xlim(-max_display_r, max_display_r)
        ax.set_ylim(0, max_display_z)
        ax.set_aspect('equal')
        ax.set_xlabel('x (mm)')
        ax.set_ylabel('z (mm)')
        ax.set_title(f'Concentric Coils: {num_layers_outer} outer + {num_layers_inner} inner layers')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Magnetic field plots with uniformity analysis - FIX: calculate separately
        uniformity_zone = uniformity_zone_slider.value
        z_points_mm = np.linspace(-50, 50, 500)

        # Calculate outer and inner contributions separately to avoid cartesian product
        Bz_anti = np.zeros_like(z_points_mm)
        Bz_helm = np.zeros_like(z_points_mm)

        if len(radii_outer) > 0 and len(z_positions_outer) > 0:
            Bz_anti += calculate_field_on_axis(radii_outer, z_positions_outer, current, z_points_mm, 'antihelmholtz')
            Bz_helm += calculate_field_on_axis(radii_outer, z_positions_outer, current, z_points_mm, 'helmholtz')
        if len(radii_inner) > 0 and len(z_positions_inner) > 0:
            Bz_anti += calculate_field_on_axis(radii_inner, z_positions_inner, current, z_points_mm, 'antihelmholtz')
            Bz_helm += calculate_field_on_axis(radii_inner, z_positions_inner, current, z_points_mm, 'helmholtz')

        # Calculate metrics
        anti_gradient = calculate_gradient_in_zone(z_points_mm, Bz_anti, uniformity_zone)
        helm_uniformity = calculate_uniformity_in_zone(Bz_helm, uniformity_zone, z_points_mm)

        # Plot (keep original plot_field_comparison unchanged)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

        # Anti-Helmholtz plot
        ax1.plot(z_points_mm, Bz_anti, 'b-', lw=2, label='Anti-Helmholtz')
        ax1.axvline(x=uniformity_zone, color='green', lw=1, linestyle='--')
        ax1.axvline(x=-uniformity_zone, color='green', lw=1, linestyle='--')
        ax1.fill_betweenx([min(Bz_anti), max(Bz_anti)], -uniformity_zone, uniformity_zone,
                        color='green', alpha=0.1, label=f'Zone (±{uniformity_zone}mm)')
        ax1.axhline(0, color='black', lw=0.5)
        ax1.axvline(0, color='black', lw=0.5)
        ax1.set_xlabel('z (mm)')
        ax1.set_ylabel('Bz (G)')
        ax1.set_title(f'Anti-Helmholtz: Gradient = {anti_gradient:.2f} G/cm')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Helmholtz plot
        ax2.plot(z_points_mm, Bz_helm, 'r-', lw=2, label='Helmholtz')
        ax2.axvline(x=uniformity_zone, color='green', lw=1, linestyle='--')
        ax2.axvline(x=-uniformity_zone, color='green', lw=1, linestyle='--')
        ax2.fill_betweenx([min(Bz_helm), max(Bz_helm)], -uniformity_zone, uniformity_zone,
                        color='green', alpha=0.1, label=f'Zone (±{uniformity_zone}mm)')
        ax2.axhline(0, color='black', lw=0.5)
        ax2.axvline(0, color='black', lw=0.5)
        ax2.set_xlabel('z (mm)')
        ax2.set_ylabel('Bz (G)')
        ax2.set_title(f'Helmholtz: Uniformity = {helm_uniformity:.4f}% max deviation')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("\n=== Magnetic Field Analysis ===")
        print(f"Uniformity zone: ±{uniformity_zone} mm")
        print(f"Anti-Helmholtz gradient in zone: {anti_gradient:.2f} G/cm")
        print(f"Helmholtz field uniformity in zone: {helm_uniformity:.4f}% max deviation")

        # Store geometry for intensity variation
        global last_geometry
        last_geometry = (radii_outer, z_positions_outer, radii_inner, z_positions_inner, thickness, wire_section)
        intensity_current_slider.value = current  # Sync the sliders


# Interactive widget to explore the configuration
target_gradient_slider = w.FloatSlider(value=30.0, min=5.0, max=50.0, step=1.0,
                                      description='Target gradient (G/cm)',
                                      style={'description_width': 'initial'})
min_r_slider = w.FloatSlider(value=80.0, min=0.0, max=100.0, step=0.5,
                             description='Min radius (mm)',
                             style={'description_width': 'initial'})
max_r_slider = w.FloatSlider(value=102.0, min=0.0, max=120.0, step=0.5,
                             description='Max radius (mm)',
                             style={'description_width': 'initial'})
thickness_slider = w.FloatSlider(value=3.0, min=0.5, max=5.0, step=0.1,
                                description='Coil thickness (mm)',
                                style={'description_width': 'initial'})
spacing_slider = w.FloatSlider(value=100.0, min=10.0, max=200.0, step=1.0,
                              description='Spacing (μm)',
                              style={'description_width': 'initial'})
current_slider = w.FloatSlider(value=50.0, min=1.0, max=200.0, step=1.0,
                              description='Current per coil (A)',
                              style={'description_width': 'initial'})
min_z_slider = w.FloatSlider(value=41.0, min=30.0, max=150.0, step=0.5,
                             description='Min z position (mm)',
                             style={'description_width': 'initial'})
wire_section_slider = w.FloatSlider(value=2.6, min=0.1, max=10.0, step=0.1,
                                  description='Wire section (mm²)',
                                  style={'description_width': 'initial'})
uniformity_zone_slider = w.FloatSlider(value=5.0, min=1.0, max=20.0, step=0.5,
                                     description='Uniformity zone (±mm)',
                                     style={'description_width': 'initial'})
layer_adjustment_slider = w.IntSlider(value=0, min=-3, max=3, step=1,
                                     description='Layer adjustment',
                                     style={'description_width': 'initial'})
z_screw_heads_slider = w.FloatSlider(value=66.0, min=30.0, max=100.0, step=0.5,
                                     description='Z Screw Heads (mm)',
                                     style={'description_width': 'initial'})
r_min_inner_slider = w.FloatSlider(value=50.0, min=0.0, max=100.0, step=0.5,
                                     description='Min Inner Radius (mm)',
                                     style={'description_width': 'initial'})
support_slider = w.FloatSlider(value=3.0, min=0.5, max=5.0, step=0.1,
                               description='Support (mm)',
                               style={'description_width': 'initial'})

# Create Run button and output
run_button = w.Button(description="Run Calculation", button_style='success')
layers_out = w.Output()

# Add a separate slider for intensity variation
intensity_current_slider = w.FloatSlider(
    value=50.0, min=1.0, max=200.0, step=0.5,
    description='Current (A) - Vary Intensity',
    continuous_update=False
)
intensity_out = w.Output()


# Connect button click to calculation
def on_run_click(b):
    update_layers(
        target_gradient_slider.value,
        min_r_slider.value,
        max_r_slider.value,
        thickness_slider.value,
        spacing_slider.value,
        current_slider.value,
        min_z_slider.value,
        wire_section_slider.value,
        layer_adjustment_slider.value,
        z_screw_heads_slider.value,
        r_min_inner_slider.value,
        support_slider.value
    )

run_button.on_click(on_run_click)

# Observer to intensity slider
intensity_current_slider.observe(update_intensity, names='value')

# Display all widgets in a vertical layout
display(w.VBox([
    target_gradient_slider,
    min_r_slider,
    max_r_slider,
    thickness_slider,
    spacing_slider,
    current_slider,
    min_z_slider,
    uniformity_zone_slider,
    wire_section_slider,
    layer_adjustment_slider,
    z_screw_heads_slider,
    r_min_inner_slider,
    support_slider,
    run_button,
    layers_out,
    w.HTML("<h3>Intensity Variation (Fixed Geometry)</h3>"),
    intensity_current_slider,
    intensity_out
]))

## Practical Considerations

### Coil Geometry Notes

1. **d/R = 1 (Ideal anti-Helmholtz)**: This is the most common configuration, providing a good balance between gradient strength and field uniformity near the center.

2. **d/R > 1**: Larger separation produces a weaker gradient but extends the region of linear field. Useful for larger MOTs.

3. **d/R < 1**: Smaller separation produces a stronger gradient but with a smaller linear region. Useful for small, tightly confined traps.

4. **Multiple coil pairs**: For more uniform gradients over larger volumes, multiple anti-Helmholtz coil pairs can be used in series.

### Cooling

Anti-Helmholtz coils can dissipate significant power as heat. Consider:
* **Air cooling**: Sufficient for most small-scale MOTs (< 100 W)
* **Water cooling**: Necessary for high-power applications (< 3kW)
* **Pulsed operation**: Can reduce average power requirements

### References

* [Copper Resistivity](http://hyperphysics.phy-astr.gsu.edu/hbase/Tables/rstiv.html)
* [Magneto-Optical Trap Review](https://arxiv.org/abs/physics/9803027)

## Summary

This notebook provides comprehensive tools for designing and analyzing anti-Helmholtz coil configurations for atomic physics applications, particularly Magneto-Optical Traps (MOTs).

### Key Features:
* Interactive visualization of magnetic field profiles
* Comparison with Helmholtz coil configurations
* 2D field mapping
* MOT-specific parameter calculations
* Design tools for achieving target gradients
* Power optimization for given constraints

### Next Steps:
* Integrate with specific experimental parameters
* Consider thermal effects and cooling requirements
* Add field compensation coil calculations
* Implement more accurate off-axis field calculations using elliptic integrals